# Dataset Analysis

## Utils 

In [6]:
import igraph as ig 
from typing import Tuple
import numpy as np
import subprocess
import pandas as pd
import time as time
import random

In [2]:
def import_graph(file_path: str) -> ig.Graph:
    """
    Import a graph from a txt file using igraph

    Parameters
    ----------
    file_path : str
        File path of the .txt file
        
    Returns
    -------
    ig.Graph
        Graph imported from the file path
    """
    if file_path.endswith(".txt"):
        graph = ig.Graph.Read_Edgelist(file_path, directed=False)
        graph = graph.simplify(multiple=True, loops=True)
    else:
        raise ValueError("File format not supported")
    
    # Remove nodes with degree 0
    ## Igraph starts from 0 index, so if txt file starts from 1, we need to delete the first vertex
    graph.delete_vertices(graph.vs.select(_degree_eq=0))
    return graph

In [3]:
repo_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).strip().decode('utf-8')
dataset_root = f"{repo_root}/dataset/networks"

In [4]:
def graph_analysis(G: ig.Graph, graph_name: str) -> None:
    """
    Perform some analysis on the graph

    Parameters
    ----------
    G : ig.Graph
        Graph to analyze
    """
    networks_names = {
        "kar": "kar - Zachary Karate Club",
        "words": "words - David Copperfield Word Adjacency Network",
        "vote": "vote - Wikipedia Voting Network",
        "pow": "pow - U.S. Power Grid",
        "fb-75": "fb-75 - Facebook Friendship Network",
        "cond-mat": "cond-mat - Condense Matter Collaboration Network",
        "warpcast_500": "warp_500 - Warpcast Network",
        "warpcast_1k": "warp_1k - Warpcast Network",
        "warpcast_2k": "warp_2k - Warpcast Network",
        "warpcast_5k": "warp_5k - Warpcast Network",
        "warpcast_10k": "warp_10k - Warpcast Network",
        "warpcast_20k": "warp_20k - Warpcast Network",
        "warpcast_50k": "warp_50k - Warpcast Network",
        "warpcast_100k": "warp_100k - Warpcast Network",
        "warpcast_200k": "warp_200k - Warpcast Network",
    }

    #Graph Analysis
    n_nodes = G.vcount()
    n_edges = G.ecount()
    degrees = G.degree()
    diameter = G.diameter()
    avg_degree = sum(degrees) / len(degrees)
    max_degree = max(degrees)
    avg_path_length = G.average_path_length()

    #Community Detection Analysis
    algorithms = {
        "Edge Betweenness": G.community_edge_betweenness,
        "Fast Greedy": G.community_fastgreedy,
        "Infomap": G.community_infomap,
        "Label Propagation": G.community_label_propagation,
        "Leading Eigenvector": G.community_leading_eigenvector,
        "Louvain": G.community_multilevel,
        "Spinglass": G.community_spinglass,
        "Walktrap": G.community_walktrap
    }
    results = []
    for name, algorithm in algorithms.items():
        #Avoid running some time-consuming algorithms on large graphs
        if n_nodes > 200 and name == "Edge Betweenness":
            continue
        elif n_nodes > 900 and name == "Spinglass":
            continue
        elif n_nodes > 10000 and name == "Walktrap":
            continue
        start_time = time.time()
        random.seed(22)
        try:
            if name in ["Edge Betweenness", "Fast Greedy", "Walktrap"]:
                communities = algorithm().as_clustering()
            else:
                communities = algorithm()
            end_time = time.time()
            elapsed_time = round(end_time - start_time, 2)
            results.append({"Algorithm": name, "Number of Communities": len(communities), "Time (s)": elapsed_time})
        except ig.InternalError as e:
            print(f"Algorithm {name} failed: {e}")
            results.append({"Algorithm": name, "Number of Communities": "N/A", "Time (s)": "N/A"})
    df = pd.DataFrame(results)

    print("-"*20,"Graph Analysis","-"*20,"\n")
    print(f"Graph: {networks_names[graph_name]}")
    print(f"Number of nodes: {n_nodes}")
    print(f"Number of edges: {n_edges}")
    print(f"Average degree: {avg_degree:.2f}")
    print(f"Max degree: {max_degree}")
    print(f"Diameter: {diameter}")
    print(f"Average path length: {avg_path_length:.2f} \n")
    print("-"*13,"Community Detection Analysis","-"*13,"\n")
    print(df)

## Zachary Karate Club

In [7]:
graph_name = "kar"
kar = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(kar, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: kar - Zachary Karate Club
Number of nodes: 34
Number of edges: 78
Average degree: 4.59
Max degree: 17
Diameter: 5
Average path length: 2.41 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0     Edge Betweenness                      5      0.00
1          Fast Greedy                      3      0.00
2              Infomap                      3      0.00
3    Label Propagation                      2      0.00
4  Leading Eigenvector                      4      0.01
5              Louvain                      4      0.00
6            Spinglass                      4      0.11
7             Walktrap                      5      0.00


## David Copperfield Word Adjacency Network

In [8]:
graph_name = "words"
words = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(words, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: words - David Copperfield Word Adjacency Network
Number of nodes: 112
Number of edges: 425
Average degree: 7.59
Max degree: 49
Diameter: 5
Average path length: 2.54 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0     Edge Betweenness                     69      0.13
1          Fast Greedy                      7      0.00
2              Infomap                      2      0.01
3    Label Propagation                      1      0.00
4  Leading Eigenvector                     10      0.01
5              Louvain                      8      0.00
6            Spinglass                      8      0.61
7             Walktrap                     25      0.00


## Wikipedia Voting Network

In [9]:
graph_name = "vote"
vote = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(vote, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: vote - Wikipedia Voting Network
Number of nodes: 889
Number of edges: 2914
Average degree: 6.56
Max degree: 102
Diameter: 13
Average path length: 4.10 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                     14      0.01
1              Infomap                     76      0.24
2    Label Propagation                      9      0.00
3  Leading Eigenvector                      6      0.03
4              Louvain                      9      0.00
5            Spinglass                     11      4.00
6             Walktrap                     42      0.02


## U.S. Power Grid

In [11]:
graph_name = "pow"
pow = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(pow, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: pow - U.S. Power Grid
Number of nodes: 4941
Number of edges: 6594
Average degree: 2.67
Max degree: 19
Diameter: 46
Average path length: 18.99 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities   Time (s)
0          Fast Greedy                     41   0.010860
1              Infomap                    488   1.471950
2    Label Propagation                    494   0.014400
3  Leading Eigenvector                    126   3.195802
4              Louvain                     41   0.006394
5            Spinglass                     25  17.672550
6             Walktrap                    364   0.084916


## Facebook Friendship Network

In [18]:
graph_name = "fb-75"
fb = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(fb, graph_name)

Algorithm Spinglass failed: Error at src/community/spinglass/clustertool.cpp:293: Cannot work with unconnected graph. -- Invalid value
-------------------- Graph Analysis -------------------- 

Graph: fb-75 - Facebook Friendship Network
Number of nodes: 6386
Number of edges: 217662
Average degree: 68.17
Max degree: 930
Diameter: 9
Average path length: 2.77 

------------- Community Detection Analysis ------------- 

             Algorithm Number of Communities Time (s)
0          Fast Greedy                    24     1.16
1              Infomap                   140    10.15
2    Label Propagation                    17     0.04
3  Leading Eigenvector                    13     0.67
4              Louvain                    18     0.21
5            Spinglass                   N/A      N/A
6             Walktrap                   357     6.19


## Condense Matter Collaboration Network

In [22]:
graph_name = "cond-mat"
cond_mat = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(cond_mat, graph_name)

/Users/silver22/anaconda3/lib/python3.11/site-packages/igraph/community.py:98: RuntimeWarning: ARPACK solver failed to converge (10001 iterations, 0/1 eigenvectors converged) at src/linalg/arpack.c:899
  membership, _, q = GraphBase.community_leading_eigenvector(graph, clusters, **kwds)


Algorithm Leading Eigenvector failed: Error at src/community/leading_eigenvector.c:563: ARPACK did not converge. -- No eigenvalues to sufficient accuracy
-------------------- Graph Analysis -------------------- 

Graph: cond-mat - Condense Matter Collaboration Network
Number of nodes: 108299
Number of edges: 93439
Average degree: 1.73
Max degree: 279
Diameter: 15
Average path length: 5.35 

------------- Community Detection Analysis ------------- 

             Algorithm Number of Communities Time (s)
0          Fast Greedy                 86002     1.93
1              Infomap                 86975    45.04
2    Label Propagation                 87261     0.33
3  Leading Eigenvector                   N/A      N/A
4              Louvain                 85786      0.2
5             Walktrap                 88072    15.74


## Warpcast

### Warpcast-500

In [47]:
graph_name = "warpcast_500"
warp_500 = import_graph(f"{dataset_root}/warpcast/{graph_name}.txt")
graph_analysis(warp_500, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: warp_500 - Warpcast Network
Number of nodes: 494
Number of edges: 27070
Average degree: 109.60
Max degree: 466
Diameter: 4
Average path length: 1.80 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                      3      0.01
1              Infomap                      1      0.06
2    Label Propagation                      1      0.00
3  Leading Eigenvector                      3      0.03
4              Louvain                      4      0.01
5            Spinglass                      5     13.41
6             Walktrap                    153      0.06


### Warpcast-1k

In [50]:
graph_name = "warpcast_1k"
warp_1k = import_graph(f"{dataset_root}/warpcast/{graph_name}.txt")
graph_analysis(warp_1k, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: warp_1k - Warpcast Network
Number of nodes: 985
Number of edges: 68723
Average degree: 139.54
Max degree: 923
Diameter: 4
Average path length: 1.88 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                      4      0.04
1              Infomap                      1      0.15
2    Label Propagation                      1      0.00
3  Leading Eigenvector                      3      0.09
4              Louvain                      4      0.02
5             Walktrap                      1      0.25


### Warpcast-2k

In [51]:
graph_name = "warpcast_2k"
warp_2k = import_graph(f"{dataset_root}/warpcast/{graph_name}.txt")
graph_analysis(warp_2k, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: warp_2k - Warpcast Network
Number of nodes: 1956
Number of edges: 153756
Average degree: 157.21
Max degree: 1876
Diameter: 4
Average path length: 1.93 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                      4      0.14
1              Infomap                      1      0.34
2    Label Propagation                      1      0.01
3  Leading Eigenvector                      3      0.21
4              Louvain                      3      0.06
5             Walktrap                      1      1.03


### Warpcast-5k

In [52]:
graph_name = "warpcast_5k"
warp_5k = import_graph(f"{dataset_root}/warpcast/{graph_name}.txt")
graph_analysis(warp_5k, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: warp_5k - Warpcast Network
Number of nodes: 4814
Number of edges: 428889
Average degree: 178.18
Max degree: 4026
Diameter: 5
Average path length: 2.07 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                      8      1.38
1              Infomap                      5      1.16
2    Label Propagation                      1      0.02
3  Leading Eigenvector                      3      0.88
4              Louvain                      5      0.21
5             Walktrap                     80      7.11


### Warpcast-10k

In [53]:
graph_name = "warpcast_10k"
warp_10k = import_graph(f"{dataset_root}/warpcast/{graph_name}.txt")
graph_analysis(warp_10k, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: warp_10k - Warpcast Network
Number of nodes: 9517
Number of edges: 866436
Average degree: 182.08
Max degree: 7874
Diameter: 5
Average path length: 2.14 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                      8      5.76
1              Infomap                      7      2.94
2    Label Propagation                      1      0.04
3  Leading Eigenvector                      2      0.92
4              Louvain                      5      0.38
5             Walktrap                    714     30.93


### Warpcast-20k

In [11]:
graph_name = "warpcast_20k"
warp_20k = import_graph(f"{dataset_root}/warpcast/{graph_name}.txt")
graph_analysis(warp_20k, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: warp_20k - Warpcast Network
Number of nodes: 19425
Number of edges: 2058999
Average degree: 211.99
Max degree: 15365
Diameter: 5
Average path length: 2.17 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                     26     27.92
1              Infomap                     15     26.29
2    Label Propagation                      2      0.12
3  Leading Eigenvector                      2      3.21
4              Louvain                      7      0.85


### Warpcast-50k

In [12]:
graph_name = "warpcast_50k"
warp_50k = import_graph(f"{dataset_root}/warpcast/{graph_name}.txt")
graph_analysis(warp_50k, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: warp_50k - Warpcast Network
Number of nodes: 21937
Number of edges: 2306210
Average degree: 210.26
Max degree: 17044
Diameter: 8
Average path length: 2.21 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                     25     38.80
1              Infomap                     55     75.12
2    Label Propagation                      5      0.17
3  Leading Eigenvector                      3      8.88
4              Louvain                      9      1.54


### Warpcast-100k

In [13]:
graph_name = "warpcast_100k"
warp_100k = import_graph(f"{dataset_root}/warpcast/{graph_name}.txt")
graph_analysis(warp_100k, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: warp_100k - Warpcast Network
Number of nodes: 21953
Number of edges: 2307532
Average degree: 210.22
Max degree: 17057
Diameter: 8
Average path length: 2.21 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                     25     41.34
1              Infomap                     54     62.87
2    Label Propagation                      5      0.19
3  Leading Eigenvector                      3      8.10
4              Louvain                      9      1.14


### Warpcast-200k

In [14]:
graph_name = "warpcast_200k"
warp_200k = import_graph(f"{dataset_root}/warpcast/{graph_name}.txt")
graph_analysis(warp_200k, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: warp_200k - Warpcast Network
Number of nodes: 31548
Number of edges: 3328244
Average degree: 211.00
Max degree: 24813
Diameter: 7
Average path length: 2.22 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                     13     75.65
1              Infomap                    100    206.75
2    Label Propagation                      8      0.26
3  Leading Eigenvector                      5      9.59
4              Louvain                     13      1.57
